<center>
<h1 style="font-family:verdana">
 💻 🧑 Classificació d'intencions (PyTorch) 🧑 💻


<p> 🎯 <b>Objectiu</b>: en aquesta pràctica aprendrem a detectar la intenció de l'usuari a partir d'interaccions reals amb un xatbot. En el context de xatbots, la classificació d'intencions ajuda a entendre quina acció o resposta hauria de prendre el sistema en funció de la consulta de l'usuari.  


<p> ✨ <b>Contingut</b>: en primer lloc, farem servir una base de dades amb oracions d'interaccions en espanyol etiquetades com 19 intencions diferents. En segon lloc, realitzarem el preprocessament de les dades, és a dir, transformarem les dades perquè tinguen un format adequat per a ser introduïdes al model. I finalment, dissenyarem i entrenarem el model de classificació per detectar automàticament la intenció de les oracions.</p>  


<p> ✏ <b>Exercicis</b>: en cada secció anireu trobant exercicis que haureu d'anar resolent. </p>



---

<h2> Índex </h2>

1. [Inspecció del conjunt de dades](#section-one)
  * [Exercici 1](#ex-one)
2. [Preprocessament de dades](#section-two)
  * [Exercici 2](#ex-two)
  * [Exercici 3](#ex-three)
3. [Disseny del model i entrenament](#section-three)
  * [Exercici 4](#ex-four)
  * [Exercici 5](#ex-five)
4. [Lliurable](#section-four)
---

In [1]:
import random
import re
from collections import Counter
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.preprocessing import LabelEncoder

In [2]:
!pip install gdown
!gdown "https://drive.google.com/uc?id=1u2wzXvsuscLeFHwXcDwMDaNDy0u_99-t"
!tar -zxf nlu_ATIS_data.tar.gz

Downloading...
From: https://drive.google.com/uc?id=1u2wzXvsuscLeFHwXcDwMDaNDy0u_99-t
To: /content/nlu_ATIS_data.tar.gz
100% 122k/122k [00:00<00:00, 5.39MB/s]


<h1><a name="section-one"> 1. Inspecció del conjunt de dades </a></h1>

A la carpeta `data` tenim els diferents fitxers CSV que utilitzarem per a aquesta pràctica.

En primer lloc, llegirem les dades dels fitxers CSV amb `pandas`.

In [3]:
train_data = pd.read_csv('./data/train.csv', header=None)
val_data = train_data.tail(900)
train_data = pd.read_csv('./data/train.csv', header=None, nrows=4078)
test_data = pd.read_csv('./data/test.csv', header=None)

print('Training size:', len(train_data))
print('Validation dataset size:', len(val_data))
print('Test dataset size:', len(test_data))

Training size: 4078
Validation dataset size: 900
Test dataset size: 893


Per a aquesta primera part de la pràctica ens centrarem en la primera columna dels arxius que correspon amb les **oracions** en anglès introduïdes per l'usuari. I en la tercera columna que correspon amb la **intenció** de cada oració, és a dir, cada oració tindrà una etiqueta.

Podeu executar la cel·la següent tantes vegades com vulgueu per veure instàncies d'aquest conjunt de dades.


In [ ]:
random_number = random.randint(0, len(train_data)-1)

train_sentences = list(train_data[0])
train_labels = list(s.replace('"', '') for s in train_data[2])
train_labels = list(s.replace(' ', '') for s in train_labels)

print('Sentence: ', train_sentences[random_number])
print('Intent: ', train_labels[random_number])

Sentence:  can you tell me the afternoon nonstop flights departing from atlanta to boston
Intent:  flight


A continuació analitzarem quantes etiquetes diferents hi ha al dataset i quines són.

In [ ]:
num_labels = 0
for label in set(train_labels):
  print(f'Label {num_labels}:', label.split('.')[-1])
  num_labels += 1

print(f'\nThere are a total of {num_labels} intent labels')

Label 0: flight+airfare
Label 1: ground_service
Label 2: aircraft
Label 3: airline
Label 4: city
Label 5: quantity
Label 6: airport
Label 7: ground_fare
Label 8: airfare
Label 9: flight_no
Label 10: cheapest
Label 11: airfare+flight_time
Label 12: airline+flight_no
Label 13: distance
Label 14: ground_service+ground_fare
Label 15: aircraft+flight+flight_no
Label 16: abbreviation
Label 17: restriction
Label 18: meal
Label 19: flight
Label 20: flight_time
Label 21: capacity

There are a total of 22 intent labels


<h1><a name="section-two"> 2. Preprocessament de dades </a></h1>

En primer lloc, haurem de tokenitzar les oracions. Això consisteix a convertir el text en representacions numèriques, ja que els models esperen unitats discretes.

En aquesta pràctica farem servir una tokenització senzilla en PyTorch: construirem un diccionari de vocabulari basat en la freqüència de les paraules en el conjunt d'entrenament, igual que el Tokenizer de Keras. 0 es reserva per al *padding* i 1 per a tokens desconeguts (OOV).

Vegem com queda el vocabulari.

In [ ]:
class SimpleTokenizer:
    def __init__(self, num_words=500):
        self.num_words = num_words
        self.word_index = {}
        self.index_word = {}

    def fit_on_texts(self, texts):
        words = []
        for text in texts:
            tokens = re.findall(r"\b\w+\b|'\w+", text.lower())
            words.extend(tokens)
        counts = Counter(words)
        sorted_words = sorted(counts.keys(), key=lambda w: (-counts[w], words.index(w)))

        # Indexing starting at 1 (0 reserved for padding)
        self.word_index = {w: i + 1 for i, w in enumerate(sorted_words[:self.num_words - 1])}
        self.index_word = {i: w for w, i in self.word_index.items()}

    def texts_to_sequences(self, texts):
        sequences = []
        for text in texts:
            tokens = re.findall(r"\b\w+\b|'\w+", text.lower())
            seq = [self.word_index[w] for w in tokens if w in self.word_index]
            sequences.append(seq)
        return sequences

num_words = 500
tokenizer = SimpleTokenizer(num_words=num_words)
tokenizer.fit_on_texts(train_sentences)

vocab = tokenizer.word_index
print(vocab)

{'to': 1, 'from': 2, 'flights': 3, 'the': 4, 'on': 5, 'what': 6, 'me': 7, 'flight': 8, 'boston': 9, 'show': 10, 'san': 11, 'i': 12, 'denver': 13, 'a': 14, 'francisco': 15, 'in': 16, 'and': 17, 'atlanta': 18, 'pittsburgh': 19, 'is': 20, 'dallas': 21, 'baltimore': 22, 'all': 23, 'philadelphia': 24, 'like': 25, 'are': 26, 'list': 27, 'airlines': 28, 'of': 29, 'between': 30, 'that': 31, 'washington': 32, 'leaving': 33, 'please': 34, 'pm': 35, 'morning': 36, 'would': 37, 'fly': 38, 'for': 39, 'fare': 40, 'first': 41, 'wednesday': 42, 'after': 43, 'there': 44, 'oakland': 45, "'d": 46, 'ground': 47, 'you': 48, 'does': 49, 'trip': 50, 'transportation': 51, 'class': 52, 'arriving': 53, 'cheapest': 54, 'need': 55, 'city': 56, 'round': 57, 'with': 58, 'before': 59, 'which': 60, 'available': 61, 'have': 62, 'give': 63, 'at': 64, 'fares': 65, 'american': 66, 'afternoon': 67, 'one': 68, 'want': 69, 'how': 70, 'way': 71, 'new': 72, 'dc': 73, 'nonstop': 74, 'arrive': 75, 'earliest': 76, 'york': 77, 'g

---

 <h1><a name="ex-one"><center> ✏ Exercici 1 ✏</center></a></h1>

En aquest primer exercici us demanem que donat el vocabulari anterior convertiu la llista d'oracions de la partició d'entrenament, és a dir, `train_sentences` en seqüències d'IDs.

In [ ]:
#TODO
train_sequences = tokenizer.texts_to_sequences(train_sentences)
print(train_sentences[0])
print(train_sequences[0])

i want to fly from boston at 838 am and arrive in denver at 1110 in the morning
[12, 69, 1, 38, 2, 9, 64, 416, 84, 17, 75, 16, 13, 64, 494, 16, 4, 36]


Si ho heu fet correctament hauríeu d'obtenir una llista d'IDs corresponents a les paraules de l'oració.


---
A continuació haurem d'aconseguir que totes les seqüències tinguen una longitud fixa. En PyTorch podem utilitzar funcions com `torch.nn.utils.rnn.pad_sequence` o operacions de padding manuals per ajustar la longitud de les seqüències al valor màxim de `max_sequence_length`.


In [ ]:
def pad_sequences_torch(sequences, maxlen, padding='post'):
    padded = np.zeros((len(sequences), maxlen), dtype=np.int64)
    for i, seq in enumerate(sequences):
        if len(seq) > 0:
            trunc = seq[:maxlen]
            if padding == 'post':
                padded[i, :len(trunc)] = trunc
            elif padding == 'pre':
                padded[i, -len(trunc):] = trunc
    return torch.tensor(padded, dtype=torch.long)

max_sequence_length = max(map(len, train_sequences))
train_pad_sequences = pad_sequences_torch(train_sequences, maxlen=max_sequence_length, padding='pre')
print('Padded sequence: ', train_pad_sequences[0])

Padded sequence:  tensor([  0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,  12,  69,   1,  38,   2,   9,  64, 416,  84,  17,  75,  16,  13,
         64, 494,  16,   4,  36])


---

 <h1><a name="ex-two"><center> ✏ Exercici 2 ✏</center></a></h1>

Com l'ordre de les paraules sí que importa als models que utilitzarem en aquesta pràctica, és aconsellable que el *padding* estiga al final i no al principi (`padding='post'`). Afegiu els zeros al final de la seqüència.

In [ ]:
#TODO
train_pad_sequences = pad_sequences_torch(train_sequences, maxlen=max_sequence_length, padding='post')
print('Padded sequence: ', train_pad_sequences[0])

Padded sequence:  tensor([ 12,  69,   1,  38,   2,   9,  64, 416,  84,  17,  75,  16,  13,  64,
        494,  16,   4,  36,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,   0,
          0,   0,   0,   0,   0])


---

A continuació convertirem les classes d'intencions en etiquetes numèriques utilitzant `LabelEncoder` de scikit-learn. En PyTorch, la pràctica estàndard és utilitzar índexs enter com etiquetes (ja que `nn.CrossEntropyLoss` rep els índexs de classe directament), però si necessitem un vector one-hot podem convertir-ho fàcilment amb `torch.nn.functional.one_hot`.

In [ ]:
label_encoder = LabelEncoder()
train_numerical_labels = label_encoder.fit_transform(train_labels)

print(f'Original labels: {train_labels[:5]}\n')
print(f'Encoded labels: {train_numerical_labels[:5]} \n')

Original labels: ['flight', 'flight', 'flight_time', 'airfare', 'airfare']

Encoded labels: [12 12 15  3  3] 



I a continuació convertim les etiquetes a vectors one-hot o mantenim els tensors d'enters per a PyTorch.

In [ ]:
num_classes = len(np.unique(train_numerical_labels))
train_numerical_labels_tensor = torch.tensor(train_numerical_labels, dtype=torch.long)
train_encoded_labels = torch.nn.functional.one_hot(train_numerical_labels_tensor, num_classes=num_classes).float()

print('Example: \n')
print(f'Original label: {train_labels[0]}\n')
print(f'Numerical label: {train_numerical_labels[0]}\n')
print(f'One-hot: {train_encoded_labels[0]}\n')

Example: 

Original label: flight

Numerical label: 12

One-hot: tensor([0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 1., 0., 0., 0., 0., 0.,
        0., 0., 0., 0.])



---

 <h1><a name="ex-three"><center> ✏ Exercici 3 ✏</center></a></h1>

Amb la partició de validació i test haurem de realitzar els mateixos passos. Per tant, en aquest exercici us demanem que obtingueu `val_pad_sequences`, `val_encoded_labels`, `test_pad_sequences` i `test_encoded_labels`.

In [ ]:
#TODO
val_sentences = list(val_data[0])
test_sentences = list(test_data[0])
val_sequences = tokenizer.texts_to_sequences(val_sentences)
test_sequences = tokenizer.texts_to_sequences(test_sentences)
val_pad_sequences = pad_sequences_torch(val_sequences, maxlen=max_sequence_length, padding='post')
test_pad_sequences = pad_sequences_torch(test_sequences, maxlen=max_sequence_length, padding='post')

In [ ]:
val_labels = list(val_data[2])
test_labels = list(test_data[2])

# Clean quotation marks and whitespace from val and test labels
val_labels = [s.replace('"', '').replace(' ', '') for s in val_labels]
test_labels = [s.replace('"', '').replace(' ', '') for s in test_labels]

def remove_values_and_indices(input_list, values_to_remove, other_tensor):
    indices_to_remove = [idx for idx, item in enumerate(input_list) if item in values_to_remove]
    cleaned_list = [item for item in input_list if item not in values_to_remove]
    keep_indices = [idx for idx in range(len(input_list)) if idx not in indices_to_remove]
    cleaned_other_tensor = other_tensor[keep_indices]
    return cleaned_list, cleaned_other_tensor

values_to_remove = ['day_name','airfare+flight','flight+airline','flight_no+airline']
val_labels, val_pad_sequences = remove_values_and_indices(val_labels, values_to_remove, val_pad_sequences)
test_labels, test_pad_sequences = remove_values_and_indices(test_labels, values_to_remove, test_pad_sequences)


In [ ]:
# Now encode the labels
val_num_labels = label_encoder.transform(val_labels)
test_num_labels = label_encoder.transform(test_labels)

val_encoded_labels = torch.nn.functional.one_hot(torch.tensor(val_num_labels), num_classes=num_classes).float()
test_encoded_labels = torch.nn.functional.one_hot(torch.tensor(test_num_labels), num_classes=num_classes).float()

---

<h1><a name="section-three"> 3. Disseny del model i entrenament </a></h1>

En primer lloc, anem a comprovar si hi ha GPUs disponibles utilitzant PyTorch.

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    print(f"GPU is available! Using: {torch.cuda.get_device_name(0)}")
else:
    print("GPU is not available. The model will be trained on CPU.")

GPU is available! Using: Tesla T4


---

 <h1><a name="ex-four"><center> ✏ Exercici 4 ✏</center></a></h1>

En aquest exercici haureu de dissenyar l'arquitectura del model en PyTorch. El nostre model tindrà quatre capes:

1. La primera capa serà un **embedding** (`nn.Embedding`). Assignarà a cada índex de cada paraula un vector dens de mida *embedding_dim*.
2. La segona capa serà un **global max pooling** layer. Prendrà el valor màxim al llarg de la dimensió de la seqüència, transformant l'entrada de (batch_size, sequence_length, embedding_dim) a (batch_size, embedding_dim).
3. La tercera capa serà una capa **densa** (`nn.Linear`) amb activació ReLU (`nn.ReLU`).
4. L'última capa serà una capa **densa** final (`nn.Linear`) per a produir els logits de les classes (en PyTorch, `nn.CrossEntropyLoss` inclou automàticament la funció Softmax).

📢 Podeu consultar la documentació de PyTorch [aquí](https://pytorch.org/docs/stable/nn.html).


In [ ]:
#TODO
class IntentClassifierPyTorch(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, num_classes):
        super(IntentClassifierPyTorch, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=0)
        self.fc1 = nn.Linear(embedding_dim, hidden_dim)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_dim, num_classes)

    def forward(self, x):
        # x: (batch_size, sequence_length)
        embedded = self.embedding(x)  # (batch_size, seq_len, embedding_dim)
        pooled, _ = torch.max(embedded, dim=1)  # (batch_size, embedding_dim)
        out = self.fc1(pooled)
        out = self.relu(out)
        out = self.fc2(out)
        return out


# Training setup
embedding_dim = 64
hidden_dim = 64
vocab_size = num_words + 1
batch_size = 32
epochs = 2

train_dataset = TensorDataset(train_pad_sequences, torch.tensor(train_numerical_labels, dtype=torch.long))
val_dataset = TensorDataset(val_pad_sequences, torch.tensor(val_num_labels, dtype=torch.long))
test_dataset = TensorDataset(test_pad_sequences, torch.tensor(test_num_labels, dtype=torch.long))

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

model = IntentClassifierPyTorch(vocab_size, embedding_dim, hidden_dim, num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Training Loop
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for batch_x, batch_y in train_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader):.4f}")

# Evaluation
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for batch_x, batch_y in test_loader:
        batch_x, batch_y = batch_x.to(device), batch_y.to(device)
        outputs = model(batch_x)
        _, predicted = torch.max(outputs.data, 1)
        total += batch_y.size(0)
        correct += (predicted == batch_y).sum().item()

print(f"Test accuracy: {correct / total:.2f}")


Epoch 1/2, Loss: 1.2046
Epoch 2/2, Loss: 0.9215
Test accuracy: 0.75


Podeu veure a continuació les oracions que el model ha classificat incorrectament.

In [ ]:
model.eval()
all_preds = []
with torch.no_grad():
    for batch_x, _ in test_loader:
        batch_x = batch_x.to(device)
        outputs = model(batch_x)
        _, preds = torch.max(outputs, 1)
        all_preds.extend(preds.cpu().numpy())

predicted_labels = label_encoder.inverse_transform(all_preds)

for i in range(0, len(predicted_labels)):
    if test_labels[i] != predicted_labels[i]:
        print('Sentence: ', test_sentences[i])
        print('Original label: ', test_labels[i])
        print('Predicted label: ', predicted_labels[i])
        print()


Sentence:  on april first i need a ticket from tacoma to san jose departing before 7 am
Original label:  airfare
Predicted label:  flight

Sentence:  show flight and prices kansas city to chicago on next wednesday arriving in chicago by 7 pm
Original label:  flight+airfare
Predicted label:  flight

Sentence:  does tacoma airport offer transportation from the airport to the downtown area
Original label:  ground_service
Predicted label:  flight

Sentence:  i need a flight from tampa to milwaukee
Original label:  meal
Predicted label:  flight

Sentence:  i need a flight from milwaukee to seattle
Original label:  meal
Predicted label:  flight

Sentence:  please find a flight from orlando to kansas city
Original label:  airport
Predicted label:  flight

Sentence:  which flights are available from tacoma to san jose in the morning on april twenty second
Original label:  airfare
Predicted label:  flight

Sentence:  which flights are available from san jose to nashville leaving in the morning 

---

 <h1><a name="ex-five"><center> ✏ Exercici 5 ✏ </center></a></h1>

Modifiqueu els següents paràmetres del model anterior i analitzeu com afecten a la seva *accuracy*:

 1. **Preprocessament.** Modifiqueu el Tokenizer per canviar la mida del vostre vocabulari i afegiu nous passos de preprocessament. Alguns possibles canvis són canviar la mida del vocabulari, treure la capitalització o fer servir *lemmatització* o *stemming*.

 2. **Mida dels Embeddings.** Proveu diferents mides d'*Embeddings* i observeu com canvia l'*accuracy* del model. Heu d'explicar les vostres conclusions.

 3. **Xarxes Convolucionals.** Afegiu capes convolucionals al vostre model (`nn.Conv1d`). Expliqueu amb detall els valors que heu provat i la vostra motivació a l'hora d'escollir-los. Recordeu, que també podeu provar diferents configuracions de *pooling* (`nn.MaxPool1d`, `nn.AdaptiveAvgPool1d`).

 4. **Xarxes Recurrents.** Afegiu capes recurrents al vostre model (`nn.LSTM`, `nn.GRU`). Expliqueu amb detall els valors que heu provat i la vostra motivació.

 5. **Regularització.** Quan proveu configuracions amb més paràmetres veureu que el model comença a tenir *overfitting* molt prompte durant l'entrenament. Afegiu *Dropout* (`nn.Dropout`) al vostre model. Heu d'explicar la vostra decisió de valors i de posició dins de la xarxa.

 6. **Balancejat de les classes.** Si analitzeu el dataset, veureu que la freqüència de les classes està molt desbalancejada. En PyTorch podeu afegir un pes per a cada classe utilitzant el paràmetre `weight` a `nn.CrossEntropyLoss(weight=class_weights)`. Calculeu un pes per a cada classe i afegiu-lo durant l'entrenament.

 ---


<h1><a name="section-four"> 4. Lliurable </a></h1>

Heu d'entregar un document PDF de com a **màxim 10 pàgines** que incloga els resultats de tots els exercicis així com una explicació de cadascun dels resultats i de la modificació que heu fet. L'estructura del document és:

1. Introducció.
2. Experiments i Resultats (amb raonament).
3. Conclusions.

No cal que afegiu el vostre codi al document, podeu entregar el *notebook* juntament amb el document.

 ---

Podemos probar de añadir un transfromer, no es obligatorrio, en la siguiente parte sí.